# <font color="#418FDE" size="6.5" uppercase>**Inspection Displays**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Use partial dependence and ICE plots to inspect fitted model behavior. 
- Compute permutation feature importance and compare it with impurity importance. 
- Create scikit-learn display objects for evaluation and inspection. 


## **1. Dependence Plots**

### **1.1. Partial Dependence**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_B/image_01_01.jpg?v=1784040647" width="250">



>* Shows predictions as one feature changes
>* Reveals average model patterns visually

>* Average predictions across chosen feature values
>* Shows typical patterns, not individual behavior

>* Shows model behavior, not real-world truth
>* Useful, but beware unrealistic feature combinations



In [ ]:
#@title Python Code - Partial Dependence

# This example visualizes average model behavior.
# Partial dependence changes one feature systematically.
# The plot shows predicted diabetes progression.

import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import train_test_split

# Load a small packaged regression dataset.
diabetes = load_diabetes(as_frame=True)
features = diabetes.data
target = diabetes.target

# Check that the selected feature exists.
feature_name = "bmi"
if feature_name not in features.columns:
    raise ValueError("Expected feature was not found.")

# Split data before fitting the model.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=42
)

# Fit one flexible model for inspection.
model = RandomForestRegressor(n_estimators=80, random_state=42)
model.fit(X_train, y_train)

# Print concise context for the display.
score = model.score(X_test, y_test)
print("scikit-learn version:", sklearn.__version__)
print("Test R-squared:", round(score, 3))
print("Inspecting feature:", feature_name)

# Create one partial dependence display.
fig, ax = plt.subplots(figsize=(7, 4))
PartialDependenceDisplay.from_estimator(
    model, X_train, [feature_name], ax=ax, kind="average"
)

# Label the plot for beginner interpretation.
ax.set_title("Partial Dependence for BMI")
ax.set_xlabel("BMI feature value")
ax.set_ylabel("Average predicted progression")
plt.tight_layout()
plt.show()



### **1.2. ICE Plots**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_B/image_01_02.jpg?v=1784040648" width="250">



>* Show individual prediction changes across feature values
>* Reveal differences hidden by average dependence plots

>* ICE lines reveal varied feature effects
>* Diverging lines suggest model interactions

>* ICE plots show model behavior, not reality
>* Use with metrics and domain knowledge



In [ ]:
#@title Python Code - ICE Plots

# This script demonstrates ICE plots for model inspection.
# ICE lines show individual prediction changes.
# The plot reveals variation hidden by averages.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import fetch_california_housing
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import train_test_split

# Try a real housing dataset, then use generated data if needed.
try:
    housing = fetch_california_housing(as_frame=True)
    data = housing.frame.sample(n=1200, random_state=42)
    feature_names = list(housing.feature_names)
    target_name = "MedHouseVal"
    if data.shape[0] != 1200 or target_name not in data.columns:
        raise ValueError("Dataset shape was not suitable.")
except Exception:
    print("Using generated regression data as an offline fallback.")
    values, target = make_regression(
        n_samples=1200,
        n_features=6,
        noise=20.0,
        random_state=42,
    )
    feature_names = ["feature_0", "feature_1", "feature_2", "feature_3", "feature_4", "feature_5"]
    data = None

# Build feature and target arrays for one regression task.
if data is not None:
    X = data[feature_names]
    y = data[target_name]
    ice_feature = "MedInc"
else:
    X = values
    y = target
    ice_feature = 0

# Split data so the fitted model is inspected on held-out rows.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
)

# Fit one small model that can learn nonlinear feature effects.
model = RandomForestRegressor(
    n_estimators=60,
    max_depth=5,
    random_state=42,
)
model.fit(X_train, y_train)

# Validate that the inspection sample is small and nonempty.
X_ice = X_test[:80]
if len(X_ice) == 0:
    raise ValueError("The ICE sample must contain at least one row.")

print("scikit-learn version:", sklearn.__version__)
print("ICE feature:", ice_feature)
print("Each thin line is one held-out observation.")

# Create one ICE display for the selected feature.
fig, ax = plt.subplots(figsize=(8, 5))
PartialDependenceDisplay.from_estimator(
    model,
    X_ice,
    features=[ice_feature],
    kind="individual",
    subsample=40,
    random_state=42,
    ax=ax,
)

ax.set_title("ICE plot: individual prediction responses")
ax.set_xlabel(str(ice_feature))
ax.set_ylabel("Predicted target")
plt.show()



### **1.3. Computation Methods**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_B/image_01_03.jpg?v=1784040650" width="250">



>* Vary one feature while holding others unchanged
>* Average predictions to reveal model response

>* ICE plots trace each observation’s prediction path
>* They reveal hidden variation and interactions

>* Choose realistic feature grids for evaluation
>* Interpret plots with context, not causality



In [ ]:
#@title Python Code - Computation Methods

# This script demonstrates partial dependence computation.
# ICE curves show individual prediction paths.
# The plot compares individual and average effects.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
import sklearn

# Create a small regression dataset with one important feature.
features, target = make_regression(
    n_samples=300,
    n_features=4,
    noise=12.0,
    random_state=42,
)

# Confirm the data has the expected beginner-friendly shape.
if features.shape != (300, 4):
    raise ValueError("Expected 300 rows and 4 columns.")

# Split data before fitting the model.
train_features, test_features, train_target, test_target = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
)

# Fit one simple tree model for inspection.
model = DecisionTreeRegressor(max_depth=4, random_state=42)
model.fit(train_features, train_target)

# Print concise context for the displayed inspection.
score = model.score(test_features, test_target)
print("scikit-learn version:", sklearn.__version__)
print("Test R-squared:", round(score, 3))
print("Inspecting feature 0 with ICE curves and their average.")

# Build one display containing ICE curves and partial dependence.
fig, ax = plt.subplots(figsize=(7, 5))
PartialDependenceDisplay.from_estimator(
    model,
    test_features,
    features=[0],
    kind="both",
    subsample=40,
    random_state=42,
    ax=ax,
)

# Label the single axes for beginner interpretation.
ax.set_title("ICE curves with average partial dependence")
ax.set_xlabel("Feature 0 value")
ax.set_ylabel("Predicted target")
plt.show()



## **2. Feature Importance**

### **2.1. Permutation Importance**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_B/image_02_01.jpg?v=1784040662" width="250">



>* Shuffle features to test performance impact
>* Larger drops indicate stronger model reliance

>* Works across many fitted model types
>* Interpret importance using task, metric, and data

>* Repeat shuffles for more stable importance estimates
>* Use validation data; importance is not causation



In [ ]:
#@title Python Code - Permutation Importance

# This example compares two feature importance methods.
# Permutation importance measures test performance loss.
# The plot highlights model reliance differences.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
feature_names = data.feature_names

# Keep the target separate from the features.
y = data.target
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target values.")

# Split data before fitting the model.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Fit one tree ensemble with built-in impurity importance.
model = RandomForestClassifier(
    n_estimators=80, max_depth=5, random_state=42
)
model.fit(X_train, y_train)

# Measure baseline accuracy on unseen test data.
baseline_accuracy = accuracy_score(y_test, model.predict(X_test))
print("scikit-learn version:", sklearn.__version__)
print("Test accuracy before shuffling:", round(baseline_accuracy, 3))

# Compute permutation importance on the test data.
perm = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=42
)

# Compare the five strongest permutation features.
order = np.argsort(perm.importances_mean)[-5:]
selected_names = feature_names[order]
perm_values = perm.importances_mean[order]
impurity_values = model.feature_importances_[order]

# Plot both importance estimates for the same features.
positions = np.arange(len(selected_names))
width = 0.38
fig, ax = plt.subplots(figsize=(9, 5))

ax.barh(positions - width / 2, perm_values, width, label="Permutation")
ax.barh(positions + width / 2, impurity_values, width, label="Impurity")
ax.set_yticks(positions)
ax.set_yticklabels(selected_names)

ax.set_xlabel("Importance value")
ax.set_ylabel("Feature")
ax.set_title("Permutation importance versus impurity importance")
ax.legend()
plt.tight_layout()
plt.show()



### **2.2. Correlated Feature Effects**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_B/image_02_02.jpg?v=1784040659" width="250">



>* Correlated features can substitute for each other
>* Permutation importance may understate shared signals

>* Related measurements can hide individual feature importance
>* Permutation importance shows model reliance, not value

>* Assess correlated features as meaningful groups
>* Treat importance rankings as interpretive clues



In [ ]:
#@title Python Code - Correlated Feature Effects

# Compare importance when predictors share information.
# Correlation can hide single feature reliance.
# The plot shows split importance patterns.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# Create one signal and two correlated copies.
rng = np.random.default_rng(42)
n_samples = 800
signal = rng.normal(size=n_samples)

# Add small noise so the first two features overlap.
feature_a = signal + rng.normal(scale=0.08, size=n_samples)
feature_b = signal + rng.normal(scale=0.08, size=n_samples)
noise_feature = rng.normal(size=n_samples)

target = 5 * signal + rng.normal(scale=0.5, size=n_samples)
X = np.column_stack([feature_a, feature_b, noise_feature])
feature_names = np.array(["correlated A", "correlated B", "noise"])

# Validate the small teaching dataset before modeling.
if X.shape != (n_samples, 3):
    raise ValueError("Expected 800 rows and 3 features.")

# Split data so importance is measured on unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    X, target, test_size=0.3, random_state=42
)

# Fit one tree ensemble with built-in impurity importance.
model = RandomForestRegressor(n_estimators=80, random_state=42)
model.fit(X_train, y_train)

# Permutation importance measures performance drop after shuffling.
perm = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=42
)

r2 = r2_score(y_test, model.predict(X_test))
print("scikit-learn version:", sklearn.__version__)
print("Test R2:", round(r2, 3))
print("Correlated features can share predictive credit.")

# Plot both importance views for the same fitted model.
x_positions = np.arange(len(feature_names))
bar_width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    x_positions - bar_width / 2, model.feature_importances_, bar_width,
    label="Impurity importance"
)
ax.bar(
    x_positions + bar_width / 2, perm.importances_mean, bar_width,
    label="Permutation importance"
)

ax.set_title("Correlated Features Split Importance")
ax.set_xlabel("Feature")
ax.set_ylabel("Importance score")
ax.set_xticks(x_positions, feature_names)
ax.legend()

plt.tight_layout()
plt.show()



### **2.3. Impurity Importance Comparison**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_B/image_02_03.jpg?v=1784040660" width="250">



>* Sums tree split purity gains by feature
>* Fast, but training-focused not validation-based

>* Shuffling features tests performance dependence
>* Validation results can reveal overfit importance

>* Impurity scores can favor high-cardinality features
>* Compare methods and investigate disagreements



In [ ]:
#@title Python Code - Impurity Importance Comparison

# Compare two tree feature importance methods.
# Impurity uses training split improvements.
# Permutation measures heldout performance drops.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = np.array(data.feature_names)

# Keep the example small and easy to read.
selected_names = np.array([
    "mean radius", "mean texture", "mean perimeter", "mean area", "mean smoothness"
])
selected_indices = [list(feature_names).index(name) for name in selected_names]
X = X[:, selected_indices]

# Validate the selected feature matrix shape.
if X.shape[1] != len(selected_names):
    raise ValueError("Feature selection did not match the expected columns.")

# Split data so permutation importance uses heldout examples.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Fit one tree ensemble with built-in impurity importance.
model = RandomForestClassifier(
    n_estimators=80, max_depth=4, random_state=42
)
model.fit(X_train, y_train)

# Compute permutation importance on the heldout test set.
permutation = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=42
)

# Put both importance scores on a comparable table.
importance_table = pd.DataFrame({
    "feature": selected_names,
    "impurity": model.feature_importances_,
    "permutation": permutation.importances_mean
})
importance_table = importance_table.sort_values("permutation", ascending=False)

print("scikit-learn version:", sklearn.__version__)
print("Test accuracy:", round(model.score(X_test, y_test), 3))
print("Top feature by impurity:", importance_table.sort_values("impurity", ascending=False).iloc[0, 0])
print("Top feature by permutation:", importance_table.iloc[0, 0])

# Plot both importance methods for the same features.
fig, ax = plt.subplots(figsize=(8, 4))
positions = np.arange(len(importance_table))
bar_width = 0.38

ax.bar(
    positions - bar_width / 2, importance_table["impurity"], bar_width,
    label="Impurity importance"
)
ax.bar(
    positions + bar_width / 2, importance_table["permutation"], bar_width,
    label="Permutation importance"
)

ax.set_title("Impurity vs Permutation Feature Importance")
ax.set_xlabel("Feature")
ax.set_ylabel("Importance score")
ax.set_xticks(positions)
ax.set_xticklabels(importance_table["feature"], rotation=25, ha="right")
ax.legend()

plt.tight_layout()
plt.show()



## **3. Display Objects**

### **3.1. Display Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_B/image_03_01.jpg?v=1784040652" width="250">



>* Reusable visuals connect models, data, and plots
>* Diagnostics support comparison, customization, and trust

>* Organize model data into customizable plots
>* Turn predictions into interpretable visual evidence

>* Support reproducible, modular model inspection
>* Turn model results into decision-ready visuals



In [ ]:
#@title Python Code - Display Basics

# This example creates one scikit-learn display object.
# Display objects store plots for later customization.
# The result is a labeled confusion matrix.

import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_iris
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Load two iris features for a simple visual display.
iris = load_iris()
feature_names = iris.feature_names[:2]
X = iris.data[:, :2]
y = iris.target

# Validate the small teaching dataset before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data so the fitted model is evaluated fairly.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=42
)

# Fit one beginner-friendly classifier.
model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X_train, y_train)

# Create a display object from the fitted estimator.
display = DecisionBoundaryDisplay.from_estimator(
    model, X_test, response_method="predict", alpha=0.35
)

# Add the test points to the display object's axes.
display.ax_.scatter(X_test[:, 0], X_test[:, 1], c=y_test, edgecolor="black")
display.ax_.set_xlabel(feature_names[0])
display.ax_.set_ylabel(feature_names[1])

# Customize the stored axes after the display is created.
display.ax_.set_title("DecisionBoundaryDisplay stores figure and axes")
print("scikit-learn version:", sklearn.__version__)
print("Display object type:", type(display).__name__)
print("Stored axes title:", display.ax_.get_title())
plt.show()



### **3.2. Classification Displays**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_B/image_03_02.jpg?v=1784040654" width="250">



>* Turn classification results into clear visual summaries
>* Inspect errors, class patterns, and thresholds

>* Confusion matrices reveal classification mistakes
>* Display objects support reusable evaluation visuals

>* Visualize threshold effects on classifier performance
>* Compare models and choose practical operating points



In [ ]:
#@title Python Code - Classification Displays

# This example creates a classification display object.
# It visualizes model errors with a confusion matrix.
# The plot shows stored display information clearly.

import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
iris = load_iris()
X = iris.data
y = iris.target

# Validate the basic dataset shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data while preserving class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Fit preprocessing and the classifier only on training data.
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=200, random_state=42)
)
model.fit(X_train, y_train)

# Print concise context for the displayed evaluation.
accuracy = model.score(X_test, y_test)
print("scikit-learn version:", sklearn.__version__)
print("Test accuracy:", round(accuracy, 3))

# Create one display object from the fitted classifier.
display = ConfusionMatrixDisplay.from_estimator(
    model, X_test, y_test, display_labels=iris.target_names, cmap="Blues"
)

# The display object stores both values and Matplotlib artists.
print("Display type:", type(display).__name__)
print("Stored matrix shape:", display.confusion_matrix.shape)

# Add a beginner-friendly title to the single axes.
display.ax_.set_title("Iris Confusion Matrix Display")
display.ax_.set_xlabel("Predicted class")
display.ax_.set_ylabel("True class")
plt.show()



### **3.3. Regression Display Tools**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_30/Lecture_B/image_03_03.jpg?v=1784040656" width="250">



>* Inspect continuous predictions with reusable display objects
>* Reveal residual patterns beyond summary scores

>* Compare predictions against true values
>* Use residuals to spot error patterns

>* Reusable displays support consistent model comparisons
>* They improve reproducibility and stakeholder trust



In [ ]:
#@title Python Code - Regression Display Tools

# This example builds a regression display object.
# PredictionErrorDisplay compares predictions with true values.
# The plot reveals model fit visually.

import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import PredictionErrorDisplay
from sklearn.model_selection import train_test_split

# Load a small packaged regression dataset.
diabetes = load_diabetes()
features = diabetes.data
target = diabetes.target

# Check that features and target align.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target values.")

# Split data before fitting the model.
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.25, random_state=42
)

# Fit one simple regression model.
model = LinearRegression()
model.fit(features_train, target_train)

# Print concise context for the display.
score = model.score(features_test, target_test)
print("scikit-learn version:", sklearn.__version__)
print("Test R-squared:", round(score, 3))

# Create one reusable scikit-learn display object.
display = PredictionErrorDisplay.from_estimator(
    model, features_test, target_test, kind="actual_vs_predicted"
)

# Customize the display through its Matplotlib axes.
display.ax_.set_title("Prediction Error Display")
display.ax_.set_xlabel("True diabetes progression")
display.ax_.set_ylabel("Predicted diabetes progression")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Inspection Displays**</font>


In this lecture, you learned to:
- Use partial dependence and ICE plots to inspect fitted model behavior. 
- Compute permutation feature importance and compare it with impurity importance. 
- Create scikit-learn display objects for evaluation and inspection. 

In the next Module (Module 31), we will go over 'Computing Parallelism'